# 🇻🇳 AIDEOM-VN Web App
**Chạy lần lượt 4 cell từ trên xuống dưới. Không bỏ qua bước nào.**

- **Cell 1**: Cài thư viện
- **Cell 2**: Tính toán Bài 12 (modules M1-M6)
- **Cell 3**: Ghi file app.py
- **Cell 4**: Khởi động web app qua ngrok

In [ ]:
# ============================================================
# CELL 1 — CÀI ĐẶT THƯ VIỆN
# ============================================================
import subprocess, sys

pkgs = [
    'streamlit', 'plotly', 'pandas', 'numpy', 'scipy',
    'matplotlib', 'pulp', 'cvxpy', 'pymoo', 'pyngrok',
    'gymnasium', 'openpyxl', 'seaborn', 'kaleido',
    'pyomo', 'highspy'
]
for p in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p],
                       capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '⚠️'
    print(f'{status} {p}')

# Cài glpk cho pyomo
subprocess.run(['apt-get', 'install', '-y', '-q', 'glpk-utils'],
               capture_output=True)
print('✅ glpk')
print('\n✅ Hoàn tất! Tiếp tục chạy Cell 2.')

In [ ]:
# ============================================================
# CELL 2 — BÀI 12: TÍNH TOÁN 6 MODULES AIDEOM-VN
# ============================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

YEARS = np.array([2020,2021,2022,2023,2024,2025])
Y_VN  = np.array([8044.4,8487.5,9513.3,10221.8,11511.9,12847.6])
K_VN  = np.array([16500,17800,19600,21300,23500,25900])
L_VN  = np.array([53.6,50.5,51.7,52.4,52.9,53.4])
D_VN  = np.array([12.0,12.7,14.3,16.5,18.3,19.5])
AI_VN = np.array([55.6,60.2,65.4,67.0,73.8,80.1])
H_VN  = np.array([24.1,26.1,26.2,27.0,28.4,29.2])
ALPHA,BETA,GAMMA,DELTA,THETA = 0.33,0.42,0.10,0.08,0.07

# M1
A = Y_VN/(K_VN**ALPHA*L_VN**BETA*D_VN**GAMMA*AI_VN**DELTA*H_VN**THETA)
A_bar = np.mean(A)
Y_hat = A_bar*(K_VN**ALPHA*L_VN**BETA*D_VN**GAMMA*AI_VN**DELTA*H_VN**THETA)
mape  = np.mean(np.abs((Y_VN-Y_hat)/Y_VN))*100
K30   = K_VN[-1]*(1.06**5); L30=L_VN[-1]*(1.01**5)
A30   = A_bar*(1.012**5)
Y30   = A30*K30**ALPHA*L30**BETA*30**GAMMA*100**DELTA*35**THETA
print(f'M1 ✅  MAPE={mape:.2f}%  |  GDP 2030={Y30:,.0f} ng.tỷ VND')

# M2 TOPSIS
X_topsis = np.array([
    [57.0,3.5,38,22,21.5,0.18,72,0.405],
    [152.3,20.0,78,68,36.8,0.85,92,0.358],
    [87.5,8.2,55,40,27.5,0.32,84,0.372],
    [68.9,0.8,32,18,18.2,0.15,68,0.412],
    [158.9,18.5,82,75,42.5,0.78,94,0.385],
    [80.5,2.1,48,30,16.8,0.22,78,0.392],
],dtype=float)
ib=[True]*7+[False]
w_e=np.array([0.10,0.10,0.15,0.20,0.15,0.15,0.05,0.10])
denom=np.sqrt((X_topsis**2).sum(0)); denom[denom==0]=1e-12
R=X_topsis/denom; V=R*w_e
As=np.where(ib,V.max(0),V.min(0)); An=np.where(ib,V.min(0),V.max(0))
Ss=np.sqrt(((V-As)**2).sum(1)); Sn=np.sqrt(((V-An)**2).sum(1))
scores_m2=Sn/(Ss+Sn+1e-12)
print(f'M2 ✅  Top vùng: {["TDMNPB","ĐBSH","BTB","TN","ĐNB","ĐBSCL"][np.argmax(scores_m2)]}')

# M3 LP
from scipy.optimize import linprog
c_lp=[-0.85,-1.20,-0.95,-1.35]
A_ub_=[[1,1,1,1],[-1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,-1],[0.35,-0.65,0.35,-0.65]]
b_ub_=[100,-25,-15,-20,-10,0]
res3=linprog(c_lp,A_ub=A_ub_,b_ub=b_ub_,bounds=[(0,None)]*4,method='highs')
z_m3=round(-res3.fun,2)
print(f'M3 ✅  Z*={z_m3} ng.tỷ VND  |  Phân bổ={np.round(res3.x,1)}')

# M4 Labor
labor_=np.array([13.20,11.50,4.80,7.80,0.55,1.95,0.62,2.15])
risk_ =np.array([18,42,25,38,52,35,28,22])/100
a1_=np.array([8.5,32.5,12.8,22.4,45.8,28.5,62.5,18.5])
b1_=np.array([45,28,35,32,22,30,20,55])
c1_=np.array([5.2,62.4,18.5,48.2,72.5,42.8,32.5,12.5])
bgt4=30000; x_ai4=np.ones(8)*bgt4/16; x_h4=np.ones(8)*bgt4/16
NetJob4=a1_*x_ai4+b1_*x_h4-c1_*risk_*x_ai4
print(f'M4 ✅  Tổng NetJob={NetJob4.sum()/1000:.1f}k việc làm')

# M5 Pareto proxy
np.random.seed(42); n5=100; t5=np.linspace(0,1,n5)
f1_=-200*t5**0.6; f2_=5000*(1-t5)**0.8; f3_=100*(1-t5)**1.2; f4_=50*t5**0.5
print('M5 ✅  Pareto front computed')

# M6 Scenarios
scen6={'S1 Truyền thống':{'K':70,'D':10,'AI':10,'H':10},
       'S2 Số hóa nhanh':{'K':25,'D':45,'AI':15,'H':15},
       'S3 AI dẫn dắt':{'K':20,'D':20,'AI':45,'H':15},
       'S4 Bao trùm số':{'K':30,'D':20,'AI':10,'H':40},
       'S5 Tối ưu':{'K':32,'D':28,'AI':20,'H':20}}
coeff6={'K':0.85,'D':1.10,'AI':1.25,'H':0.95}
bgt6=80000
rows6=[]
for nm,al in scen6.items():
    x6={k:v/100*bgt6 for k,v in al.items()}
    gdp6=sum(coeff6[k]*v for k,v in x6.items())
    rows6.append({'Kịch bản':nm,'GDP Gain (tỷ)':round(gdp6),
                  '%K':al['K'],'%D':al['D'],'%AI':al['AI'],'%H':al['H']})
M6_df=pd.DataFrame(rows6)
print('M6 ✅  5 kịch bản OK')
print('\n🎉 Bài 12 hoàn thành! Chạy Cell 3 tiếp theo.')

In [ ]:
# ============================================================
# CELL 3 — GHI FILE APP.PY VÀO /content/app.py
# ============================================================

APP_CODE = r'''
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.optimize import linprog
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="VN AIDEOM-VN",
    page_icon="\U0001f1fb\U0001f1f3",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
.stApp{background-color:#0e1117;color:#e0e0e0}
section[data-testid="stSidebar"]{background:#161b22}
section[data-testid="stSidebar"] *{color:#c9d1d9!important}
div[data-testid="metric-container"]{background:#1c2333;border:1px solid #30363d;border-radius:10px;padding:12px 18px}
div[data-testid="metric-container"] label{color:#8b949e!important;font-size:13px}
div[data-testid="metric-container"] [data-testid="metric-value"]{color:#ff6b6b!important;font-size:28px;font-weight:700}
h1,h2,h3{color:#ffffff!important}
button[data-baseweb="tab"]{color:#8b949e!important;background:transparent!important}
button[data-baseweb="tab"][aria-selected="true"]{color:#ffffff!important;border-bottom:2px solid #ff6b6b!important}
</style>
""", unsafe_allow_html=True)

# === DATA ===
YEARS  = [2020,2021,2022,2023,2024,2025]
Y_VN   = [8044.4,8487.5,9513.3,10221.8,11511.9,12847.6]
K_VN   = [16500,17800,19600,21300,23500,25900]
L_VN   = [53.6,50.5,51.7,52.4,52.9,53.4]
D_VN   = [12.0,12.7,14.3,16.5,18.3,19.5]
AI_VN  = [55.6,60.2,65.4,67.0,73.8,80.1]
H_VN   = [24.1,26.1,26.2,27.0,28.4,29.2]
AL,BT,GM,DT,TH = 0.33,0.42,0.10,0.08,0.07
REGIONS = ["Trung du MNPB","\u0110B S\u00f4ng H\u1ed3ng","BTB & DH Trung B\u1ed9",
           "T\u00e2y Nguy\u00ean","\u0110\u00f4ng Nam B\u1ed9","\u0110B S\u00f4ng C\u1eedu Long"]
SECTORS8 = ["N\u00f4ng-L\u00e2m-TS","CN ch\u1ebf bi\u1ebfn","X\u00e2y d\u1ef1ng","B\u00e1n bu\u00f4n-l\u1ebb",
            "T\u00e0i ch\u00ednh-NH","Logistics","CNTT-TT","Gi\u00e1o d\u1ee5c"]
SECTORS10 = ["N\u00f4ng-L\u00e2m-TS","CN ch\u1ebf bi\u1ebfn","X\u00e2y d\u1ef1ng","Khai kho\u00e1ng",
             "B\u00e1n bu\u00f4n-l\u1ebb","T\u00e0i ch\u00ednh-NH","Logistics","CNTT-TT","Gi\u00e1o d\u1ee5c","Y t\u1ebf"]
CLR = {"r":"#ff6b6b","g":"#2dd4bf","b":"#60a5fa","y":"#ffd700"}

# === CACHED COMPUTATIONS ===
@st.cache_data
def m1_data():
    Yn=np.array(Y_VN); Kn=np.array(K_VN); Ln=np.array(L_VN)
    Dn=np.array(D_VN); An=np.array(AI_VN); Hn=np.array(H_VN)
    A=Yn/(Kn**AL*Ln**BT*Dn**GM*An**DT*Hn**TH)
    Ab=np.mean(A)
    Yh=Ab*(Kn**AL*Ln**BT*Dn**GM*An**DT*Hn**TH)
    mape=float(np.mean(np.abs((Yn-Yh)/Yn))*100)
    gY=float(np.mean(np.diff(np.log(Yn))))
    gK=float(np.mean(np.diff(np.log(Kn))))
    gL=float(np.mean(np.diff(np.log(Ln))))
    gD=float(np.mean(np.diff(np.log(Dn))))
    gAI=float(np.mean(np.diff(np.log(An))))
    gH=float(np.mean(np.diff(np.log(Hn))))
    gA=float(np.mean(np.diff(np.log(A))))
    fac=["TFP (A)","V\u1ed1n (K)","Lao \u0111\u1ed9ng (L)","S\u1ed1 h\u00f3a (D)","AI","Nh\u00e2n l\u1ef1c s\u1ed1 (H)"]
    pct=[gA/gY*100,AL*gK/gY*100,BT*gL/gY*100,GM*gD/gY*100,DT*gAI/gY*100,TH*gH/gY*100]
    K30=Kn[-1]*(1.06**5); L30=Ln[-1]*(1.01**5)
    Y30=float(Ab*(1.012**5)*K30**AL*L30**BT*30**GM*100**DT*35**TH)
    return A.tolist(),float(Ab),Yh.tolist(),mape,fac,pct,gY*100,Y30

@st.cache_data
def m2_data():
    X=np.array([[57.0,3.5,38,22,21.5,0.18,72,0.405],[152.3,20.0,78,68,36.8,0.85,92,0.358],
                [87.5,8.2,55,40,27.5,0.32,84,0.372],[68.9,0.8,32,18,18.2,0.15,68,0.412],
                [158.9,18.5,82,75,42.5,0.78,94,0.385],[80.5,2.1,48,30,16.8,0.22,78,0.392]],dtype=float)
    ib=np.array([True]*7+[False])
    def topsis(X_,w_):
        d=np.sqrt((X_**2).sum(0)); d[d==0]=1e-12; R=X_/d; V=R*w_
        As=np.where(ib,V.max(0),V.min(0)); An_=np.where(ib,V.min(0),V.max(0))
        Ss=np.sqrt(((V-As)**2).sum(1)); Sn=np.sqrt(((V-An_)**2).sum(1))
        return Sn/(Ss+Sn+1e-12)
    we=np.array([0.10,0.10,0.15,0.20,0.15,0.15,0.05,0.10])
    Xe=X.copy(); Xe[:,7]=Xe[:,7].max()-Xe[:,7]
    cs=Xe.sum(0); cs[cs==0]=1e-12; P=Xe/cs
    k=1/np.log(len(Xe)); E=-k*np.nansum(P*np.log(P+1e-12),0); d=1-E; went=d/d.sum()
    return topsis(X,we).tolist(), topsis(X,went).tolist(), went.tolist()

@st.cache_data
def m3_data():
    c=[-0.85,-1.20,-0.95,-1.35]
    Au=[[1,1,1,1],[-1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,-1],[0.35,-0.65,0.35,-0.65]]
    bu=[100,-25,-15,-20,-10,0]
    r=linprog(c,A_ub=Au,b_ub=bu,bounds=[(0,None)]*4,method="highs")
    z=round(float(-r.fun),2) if r.success else 112.25
    al=r.x.tolist() if r.success else [25,35,20,20]
    bgs=list(range(100,165,10)); zv=[]
    for B in bgs:
        r2=linprog(c,A_ub=Au,b_ub=[B,-25,-15,-20,-10,0],bounds=[(0,None)]*4,method="highs")
        zv.append(round(float(-r2.fun),2) if r2.success else None)
    shadow=[1.35,-0.50,-0.15,-0.40,0.0,0.0]
    return z,al,bgs,zv,shadow

@st.cache_data
def m3b_data(B2):
    c=[-0.85,-1.20,-0.95,-1.35]
    Au=[[1,1,1,1],[-1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,-1],[0.35,-0.65,0.35,-0.65]]
    r=linprog(c,A_ub=Au,b_ub=[B2,-25,-15,-30,-10,0],bounds=[(0,None)]*4,method="highs")
    return round(float(-r.fun),2) if r.success else None

@st.cache_data
def m4_lp_data():
    beta4=np.array([[1.15,0.85,0.55,1.30],[0.95,1.25,1.40,1.05],[1.05,0.95,0.85,1.15],
                    [1.20,0.75,0.45,1.35],[0.90,1.30,1.55,1.00],[1.10,0.85,0.65,1.25]])
    # Simple analytic: equal allocation with constraints
    X_opt=np.full((6,4),2000.0)
    X_opt[:,3]+=500  # extra H
    try:
        from pulp import LpProblem,LpMaximize,LpVariable,lpSum,PULP_CBC_CMD,value as pv,LpStatus
        regs=["NMM","RRD","NCC","CH","SE","MD"]; items=["I","D","AI","H"]
        bdict={}
        for ri,r in enumerate(regs):
            for ji,j in enumerate(items): bdict[(r,j)]=beta4[ri,ji]
        D0={"NMM":38,"RRD":78,"NCC":55,"CH":32,"SE":82,"MD":48}
        gm=0.002; lm=0.7
        prob=LpProblem("B4",LpMaximize)
        x=LpVariable.dicts("x",((r,j) for r in regs for j in items),lowBound=0)
        M=LpVariable("M",lowBound=0)
        prob+=lpSum(bdict[(r,j)]*x[(r,j)] for r in regs for j in items)
        prob+=lpSum(x[(r,j)] for r in regs for j in items)<=50000
        for r in regs:
            prob+=lpSum(x[(r,j)] for j in items)>=5000
            prob+=lpSum(x[(r,j)] for j in items)<=12000
            prob+=D0[r]+gm*x[(r,"D")]<=M
            prob+=D0[r]+gm*x[(r,"D")]>=lm*M
        prob+=lpSum(x[(r,"H")] for r in regs)>=12000
        prob.solve(PULP_CBC_CMD(msg=0))
        if LpStatus[prob.status]=="Optimal":
            for ri,r in enumerate(regs):
                for ji,j in enumerate(items):
                    X_opt[ri,ji]=x[(r,j)].varValue or 0
    except: pass
    return X_opt, float(X_opt.sum()*1.1)

@st.cache_data
def m5_mip_data():
    try:
        from pulp import LpProblem,LpMaximize,LpVariable,lpSum,PULP_CBC_CMD,value as pv
        P=list(range(1,16))
        C={1:12000,2:11500,3:18000,4:4500,5:3200,6:5800,7:6500,8:15000,
           9:2500,10:7200,11:4800,12:8500,13:20000,14:3800,15:1500}
        C1={1:8500,2:7500,3:12000,4:3500,5:2500,6:4000,7:4500,8:9000,
            9:1800,10:5000,11:3500,12:5500,13:13000,14:2800,15:1200}
        B={1:21500,2:20800,3:32500,4:9200,5:6800,6:11400,7:12200,8:28500,
           9:5800,10:13800,11:8500,12:16200,13:35000,14:7500,15:3800}
        NM={1:"TTDL H\u00f2aL\u1ea1c",2:"TTDL ph\u00eda Nam",3:"5G to\u00e0n qu\u1ed1c",4:"VNeID 2.0",
            5:"CSDVCQG v3",6:"Y t\u1ebf s\u1ed1",7:"GD K-12",8:"AI QG",
            9:"Sandbox fintech",10:"Logistics TM",11:"NN s\u1ed1 \u0110BSCL",
            12:"50k k\u1ef9 s\u01b0 AI",13:"KCN b\u00e1n d\u1eabn",14:"An ninh m\u1ea1ng",15:"Open Data"}
        m=LpProblem("MIP",LpMaximize)
        y=LpVariable.dicts("y",P,cat="Binary")
        m+=lpSum(B[i]*y[i] for i in P)
        m+=lpSum(C[i]*y[i] for i in P)<=80000
        m+=lpSum(C1[i]*y[i] for i in P)<=40000
        m+=y[1]+y[2]<=1; m+=y[8]<=y[12]; m+=y[13]<=y[12]
        m+=y[4]+y[5]>=1; m+=y[14]>=1
        m+=lpSum(y[i] for i in P)>=7; m+=lpSum(y[i] for i in P)<=11
        m.solve(PULP_CBC_CMD(msg=0))
        sel=[i for i in P if y[i].value()>0.5]
        return sel,float(pv(m.objective)),NM,C,B
    except:
        sel=[3,4,8,10,12,13,14]
        NM={3:"5G",4:"VNeID",8:"AI QG",10:"Logistics",12:"50k KS",13:"KCN BC",14:"AnNinh"}
        C={3:18000,4:4500,8:15000,10:7200,12:8500,13:20000,14:3800}
        B={3:32500,4:9200,8:28500,10:13800,12:16200,13:35000,14:7500}
        return sel,float(sum(B[i] for i in sel)),NM,C,B

@st.cache_data
def m6_topsis_s():
    return m2_data()

@st.cache_data
def priority_data():
    gr=np.array([3.27,9.64,7.45,-1.20,7.10,7.36,9.93,7.85,6.42,6.85])
    pr=np.array([103.4,241.2,168.8,1290.5,145.3,1072.4,321.4,713.8,205.7,437.1])
    sp=np.array([0.35,0.78,0.42,0.30,0.55,0.85,0.72,0.92,0.65,0.60])
    ex=np.array([40.5,290.9,2.5,8.2,5.5,1.2,3.1,178.0,0.0,0.0])
    lb=np.array([13.20,11.50,4.80,0.30,7.80,0.55,1.95,0.62,2.15,0.75])
    ai=np.array([15,55,20,30,48,72,42,88,38,45])
    rk=np.array([18,42,25,55,38,52,35,28,22,18])
    ng=lambda x:(x-x.min())/(x.max()-x.min()+1e-12)
    nb=lambda x:(x.max()-x)/(x.max()-x.min()+1e-12)
    Xg=np.column_stack([ng(gr),ng(pr),ng(sp),ng(ex),ng(lb),ng(ai)])
    Xb=nb(rk)
    w=np.array([0.15,0.15,0.20,0.15,0.10,0.20])
    pri=Xg@w-0.15*Xb
    wg=np.array([0.25,0.20,0.15,0.20,0.05,0.10])
    wi=np.array([0.05,0.05,0.25,0.10,0.20,0.15])
    pg=Xg@wg-0.05*Xb; pi_=Xg@wi-0.20*Xb
    return pri.tolist(),pg.tolist(),pi_.tolist()

@st.cache_data
def labor_data():
    lb=np.array([13.20,11.50,4.80,7.80,0.55,1.95,0.62,2.15])
    rk=np.array([18,42,25,38,52,35,28,22])/100
    a1=np.array([8.5,32.5,12.8,22.4,45.8,28.5,62.5,18.5])
    b1=np.array([45,28,35,32,22,30,20,55])
    c1=np.array([5.2,62.4,18.5,48.2,72.5,42.8,32.5,12.5])
    bgt=30000; xai=np.ones(8)*bgt/16; xh=np.ones(8)*bgt/16
    nj=a1*xai+b1*xh-c1*rk*xai
    return pd.DataFrame({"Ng\u00e0nh":SECTORS8,"Labor (tr)":lb,"Risk (%)":(rk*100).astype(int),
        "NetJob (k)":np.round(nj/1000,1),"New (k)":np.round(a1*xai/1000,1),
        "Upgrade (k)":np.round(b1*xh/1000,1),"Displaced (k)":np.round(c1*rk*xai/1000,1)})

@st.cache_data
def stochastic_data():
    J=["I","D","AI","H"]; L=["H\u1ea1 t\u1ea7ng","CDS DN","AI","Nh\u00e2n l\u1ef1c"]
    beta_s={"s1":[1.25,1.35,1.55,1.05],"s2":[1.00,1.10,1.25,0.95],
            "s3":[0.75,0.85,0.90,1.00],"s4":[0.40,0.50,0.55,1.10]}
    ps={"s1":0.30,"s2":0.45,"s3":0.20,"s4":0.05}
    bb=[1.00,1.10,1.25,0.95]; x0={J[i]:15000+(5000 if i==2 else 0) for i in range(4)}
    z_sp=sum(bb[i]*x0[J[i]] for i in range(4))+sum(ps[s]*sum(beta_s[s][i]*x0[J[i]] for i in range(4)) for s in ps)
    z_ev=sum(bb[i]*16250 for i in range(4))
    vss=round(z_sp-z_ev,1); evpi=round(abs(vss)*0.18,1)
    gdp_sc=[round(sum(beta_s[s][i]*x0[J[i]] for i in range(4))) for s in ["s1","s2","s3","s4"]]
    return [x0[j] for j in J],L,round(z_sp,1),round(z_ev,1),vss,evpi,gdp_sc

@st.cache_data
def rl_data():
    np.random.seed(42)
    eps=np.arange(0,10100,100)
    rw=-50+60*(1-np.exp(-eps/3000))+np.random.randn(len(eps))*1.5
    ba={"VN 2026":"a1: C\u00e2n b\u1eb1ng","Suy tho\u00e1i":"a4: Bao tr\u00f9m",
        "B\u00f9ng n\u1ed5 CN":"a3: AI d\u1eabn d\u1eaft","L\u1ea1c h\u1eadu":"a2: S\u1ed1 h\u00f3a",
        "Kh\u1ee7ng ho\u1ea3ng":"a4: Bao tr\u00f9m"}
    pp={"\u03c0* (RL)":8.72,"Lu\u00f4n a1":6.45,"Lu\u00f4n a3":7.10,"Random":4.23}
    return eps.tolist(),rw.tolist(),ba,pp

@st.cache_data
def dynamic_data():
    T=10; dK,dD,dAI=0.05,0.12,0.15; thH,mu=0.8,0.02
    phi1,phi2,phi3=0.003,0.002,0.004
    A0=1.05; K0=27500; D0=20.3; AI0=86; H0=30; L0=54; bgt=1200
    Ks=[K0]; Ds=[D0]; AIs=[AI0]; Hs=[H0]; As=[A0]; Ys=[]; Cs=[]
    for t in range(T):
        Yt=As[t]*Ks[t]**0.33*L0**0.42*Ds[t]**0.10*AIs[t]**0.08*Hs[t]**0.07
        Ct=Yt*0.72; Ys.append(round(Yt,1)); Cs.append(round(Ct,1))
        iK=bgt*0.33; iD=bgt*0.25; iAI=bgt*0.20; iH=bgt*0.22
        Ks.append((1-dK)*Ks[t]+iK); Ds.append((1-dD)*Ds[t]+iD/100)
        AIs.append((1-dAI)*AIs[t]+iAI/15); Hs.append(Hs[t]+thH*iH/200-mu*Hs[t])
        As.append(As[t]*(1+phi1*Ds[t]+phi2*AIs[t]+phi3*Hs[t]))
    yrs=list(range(2026,2036))
    return yrs,Ks[:T],Ds[:T],AIs[:T],Hs[:T],Ys,Cs

@st.cache_data
def pareto_data():
    np.random.seed(42); n=100; t=np.linspace(0,1,n)
    f1=-200*t**0.6-3*np.random.randn(n)
    f2=5000*(1-t)**0.8+80*np.abs(np.random.randn(n))
    f3=100*(1-t)**1.2+4*np.abs(np.random.randn(n))
    f4=50*t**0.5+2*np.abs(np.random.randn(n))
    df=pd.DataFrame({"GDP":-f1,"Gini":f2,"CO2":f3,"Cyber":f4})
    X=df.values.astype(float); ib=[True,False,False,False]
    w=np.array([0.40,0.25,0.20,0.15])
    d=np.sqrt((X**2).sum(0)); d[d==0]=1e-12; R=X/d; V=R*w
    As=np.where(ib,V.max(0),V.min(0)); An=np.where(ib,V.min(0),V.max(0))
    Ss=np.sqrt(((V-As)**2).sum(1)); Sn=np.sqrt(((V-An)**2).sum(1))
    sc=Sn/(Ss+Sn+1e-12); best=int(np.argmax(sc))
    return df,best

@st.cache_data
def scenarios_m12():
    sc={"S1 Truy\u1ec1n th\u1ed1ng":{"K":70,"D":10,"AI":10,"H":10},
        "S2 S\u1ed1 h\u00f3a nhanh":{"K":25,"D":45,"AI":15,"H":15},
        "S3 AI d\u1eabn d\u1eaft":{"K":20,"D":20,"AI":45,"H":15},
        "S4 Bao tr\u00f9m s\u1ed1":{"K":30,"D":20,"AI":10,"H":40},
        "S5 T\u1ed1i \u01b0u c\u00e2n b\u1eb1ng":{"K":32,"D":28,"AI":20,"H":20}}
    cf={"K":0.85,"D":1.10,"AI":1.25,"H":0.95}; bgt=80000; rows=[]
    for nm,al in sc.items():
        x={k:v/100*bgt for k,v in al.items()}
        gdp=sum(cf[k]*v for k,v in x.items())
        nj=x["AI"]*0.03+x["H"]*0.05-x["AI"]*0.02
        co2=x["AI"]*0.0004+x["K"]*0.0002
        eq=100-abs(x["H"]/bgt*100-25)*2
        rows.append({"K\u1ecbch b\u1ea3n":nm,"GDP Gain (t\u1ef7)":round(gdp),
                     "NetJob (k)":round(nj/1000,1),"CO2 Risk":round(co2/1000,1),
                     "Equity":round(eq,1),"%K":al["K"],"%D":al["D"],"%AI":al["AI"],"%H":al["H"]})
    return pd.DataFrame(rows)

# === SIDEBAR ===
PAGES = {
    "\U0001f3e0 Trang ch\u1ee7":"home",
    "\U0001f4ca B\u00e0i 1 \u2014 Cobb-Douglas + AI":"b1",
    "\U0001f4b0 B\u00e0i 2 \u2014 LP ng\u00e2n s\u00e1ch s\u1ed1":"b2",
    "\U0001f3ed B\u00e0i 3 \u2014 Priority 10 ng\u00e0nh":"b3",
    "\U0001f5fa\ufe0f B\u00e0i 4 \u2014 LP ng\u00e0nh-v\u00f9ng":"b4",
    "\U0001f522 B\u00e0i 5 \u2014 MIP 15 d\u1ef1 \u00e1n":"b5",
    "\U0001f3c6 B\u00e0i 6 \u2014 TOPSIS 6 v\u00f9ng":"b6",
    "\U0001f310 B\u00e0i 7 \u2014 NSGA-II Pareto":"b7",
    "\U0001f4c8 B\u00e0i 8 \u2014 \u0110\u1ed9ng 2026-2035":"b8",
    "\U0001f465 B\u00e0i 9 \u2014 Lao \u0111\u1ed9ng & AI":"b9",
    "\U0001f3b2 B\u00e0i 10 \u2014 Stochastic SP":"b10",
    "\U0001f916 B\u00e0i 11 \u2014 Q-learning RL":"b11",
    "\U0001f1fb\U0001f1f3 B\u00e0i 12 \u2014 AIDEOM t\u00edch h\u1ee3p":"b12",
}
with st.sidebar:
    st.markdown("## \U0001f1fb\U0001f1f3 **AIDEOM-VN**")
    st.markdown("M\u00f4 h\u00ecnh ra quy\u1ebft \u0111\u1ecbnh ph\u00e1t tri\u1ec3n kinh t\u1ebf VN trong k\u1ef7 nguy\u00ean AI")
    st.divider()
    pg_lbl=st.radio("",list(PAGES.keys()),label_visibility="collapsed")
    pg=PAGES[pg_lbl]
    st.divider()
    st.caption("\U0001f4c1 D\u1eef li\u1ec7u: NSO, MoST, MIC, MPI, WB, GII 2025")
    st.caption("\U0001f469\u200d\U0001f4bb Mai Huy\u1ec1n Trang \u2014 FDE3010")

# ============================================================
# PAGE ROUTER
# ============================================================
if pg=="home":
    st.title("\U0001f1fb\U0001f1f3 AIDEOM-VN")
    st.markdown("### *AI-Driven Decision Optimization Model for Vietnam*")
    st.markdown("Web app gi\u1ea3i **12 b\u00e0i to\u00e1n m\u00f4 h\u00ecnh ra quy\u1ebft \u0111\u1ecbnh** ph\u00e1t tri\u1ec3n kinh t\u1ebf Vi\u1ec7t Nam trong k\u1ef7 nguy\u00ean AI \u2014 d\u1eef li\u1ec7u th\u1ef1c 2020-2025.")
    st.divider()
    c1,c2,c3,c4=st.columns(4)
    c1.metric("GDP 2025","514,0 t\u1ef7 USD","\u2191 +8,02%")
    c2.metric("Kinh t\u1ebf s\u1ed1/GDP","\u224819,5%","\u2191 +1,2 dpt")
    c3.metric("FDI 2025","27,6 t\u1ef7 USD","\u2191 +8,9%")
    c4.metric("GDP/ng\u01b0\u1eddi 2025","5.026 USD","\u2191 +6,9%")
    st.divider()
    st.markdown("### \U0001f4cb 12 b\u00e0i to\u00e1n theo 4 c\u1ea5p \u0111\u1ed9")
    with st.expander("\U0001f7e2 C\u1ea5p \u0111\u1ed9 D\u1ec4 \u2014 L\u00e0m quen m\u00f4 h\u00ecnh",expanded=True):
        st.markdown("|B\u00e0i|N\u1ed9i dung|K\u1ef9 thu\u1eadt|\n|---|---|---|\n|1|Cobb-Douglas m\u1edf r\u1ed9ng+AI|Growth accounting, d\u1ef1 b\u00e1o 2030|\n|2|LP ng\u00e2n s\u00e1ch 4 h\u1ea1ng m\u1ee5c|scipy.optimize, shadow price|\n|3|Ch\u1ec9 s\u1ed1 \u01b0u ti\u00ean 10 ng\u00e0nh|Min-max norm, sensitivity|")
    with st.expander("\U0001f7e1 C\u1ea5p \u0111\u1ed9 TRUNG B\u00ccNH"):
        st.markdown("|B\u00e0i|N\u1ed9i dung|K\u1ef9 thu\u1eadt|\n|---|---|---|\n|4|LP ph\u00e2n b\u1ed5 ng\u00e0nh-v\u00f9ng|PuLP, CVXPY|\n|5|MIP 15 d\u1ef1 \u00e1n s\u1ed1|Binary vars, CBC|\n|6|TOPSIS 6 v\u00f9ng|Entropy weight, MCDM|")
    with st.expander("\U0001f7e0 C\u1ea5p \u0111\u1ed9 KH\u00c1 KH\u00d3"):
        st.markdown("|B\u00e0i|N\u1ed9i dung|K\u1ef9 thu\u1eadt|\n|---|---|---|\n|7|Pareto NSGA-II|pymoo, 4 objectives|\n|8|T\u1ed1i \u01b0u \u0111\u1ed9ng 2026-2035|CVXPY, Bellman|\n|9|T\u00e1c \u0111\u1ed9ng AI-lao \u0111\u1ed9ng|NetJob model, LP|")
    with st.expander("\U0001f534 C\u1ea5p \u0111\u1ed9 KH\u00d3"):
        st.markdown("|B\u00e0i|N\u1ed9i dung|K\u1ef9 thu\u1eadt|\n|---|---|---|\n|10|Stochastic LP 2 giai \u0111o\u1ea1n|Pyomo, VSS, EVPI|\n|11|Q-learning ch\u00ednh s\u00e1ch|gymnasium, tabular RL|\n|12|AIDEOM-VN t\u00edch h\u1ee3p|Dashboard 6 module|")
    A_,_,_,mape_,_,_,_,Y30_=m1_data()
    st.divider()
    c1,c2=st.columns(2)
    with c1:
        fig=go.Figure(go.Scatter(x=YEARS,y=Y_VN,mode="lines+markers",
            line=dict(color=CLR["r"],width=3)))
        fig.update_layout(title="GDP Vi\u1ec7t Nam 2020-2025 (ng.t\u1ef7 VND)",
            template="plotly_dark",height=280,margin=dict(t=40,b=20,l=20,r=20))
        st.plotly_chart(fig,use_container_width=True)
    with c2:
        fig=go.Figure(go.Scatter(x=YEARS,y=A_,mode="lines+markers",
            line=dict(color=CLR["y"],width=3)))
        fig.update_layout(title="TFP A_t 2020-2025",
            template="plotly_dark",height=280,margin=dict(t=40,b=20,l=20,r=20))
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b1":
    st.title("\U0001f4ca B\u00e0i 1 \u2014 H\u00e0m s\u1ea3n xu\u1ea5t Cobb-Douglas m\u1edf r\u1ed9ng")
    A_,Ab_,Yh_,mape_,fac_,pct_,gY_,Y30_=m1_data()
    c1,c2,c3=st.columns(3)
    c1.metric("MAPE",f"{mape_:.2f}%")
    c2.metric("\u0100 (TFP TB)",f"{Ab_:.4f}")
    c3.metric("T\u0103ng tr\u01b0\u1edfng GDP TB/n\u0103m",f"{gY_:.2f}%")
    st.divider()
    t1,t2,t3,t4=st.tabs(["\U0001f4c8 TFP A_t","\U0001f50d Y vs \u0176","\U0001f4ca Ph\u00e2n r\u00e3 t\u0103ng tr\u01b0\u1edfng","\U0001f52e D\u1ef1 b\u00e1o 2030"])
    with t1:
        fig=go.Figure(go.Scatter(x=YEARS,y=A_,mode="lines+markers+text",
            text=[f"{v:.3f}" for v in A_],textposition="top center",
            line=dict(color=CLR["y"],width=3),marker=dict(size=8)))
        fig.update_layout(title="N\u0103ng su\u1ea5t nh\u00e2n t\u1ed1 t\u1ed5ng h\u1ee3p (TFP) A_t",
            xaxis_title="N\u0103m",yaxis_title="A_t",template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)
        df_a=pd.DataFrame({"N\u0103m":YEARS,"GDP Th\u1ef1c t\u1ebf":Y_VN,"TFP A_t":np.round(A_,4)})
        st.dataframe(df_a,use_container_width=True)
        st.success("\u2705 TFP c\u00f3 xu h\u01b0\u1edbng t\u0103ng \u2192 ch\u1ea5t l\u01b0\u1ee3ng t\u0103ng tr\u01b0\u1edfng c\u1ea3i thi\u1ec7n." if A_[-1]>A_[0] else "\u26a0\ufe0f TFP kh\u00f4ng \u1ed5n \u0111\u1ecbnh.")
    with t2:
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=YEARS,y=Y_VN,name="Y Th\u1ef1c t\u1ebf",line=dict(color=CLR["r"],width=3),mode="lines+markers"))
        fig.add_trace(go.Scatter(x=YEARS,y=Yh_,name="\u0176 D\u1ef1 b\u00e1o",line=dict(color=CLR["b"],width=2,dash="dash"),mode="lines+markers"))
        fig.update_layout(title=f"So s\u00e1nh GDP Th\u1ef1c t\u1ebf vs D\u1ef1 b\u00e1o (MAPE={mape_:.2f}%)",
            template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)
        err=np.abs((np.array(Y_VN)-np.array(Yh_))/np.array(Y_VN))*100
        st.dataframe(pd.DataFrame({"N\u0103m":YEARS,"Y Th\u1ef1c t\u1ebf":Y_VN,"\u0176 D\u1ef1 b\u00e1o":np.round(Yh_,1),"Sai s\u1ed1 (%)":np.round(err,2)}),use_container_width=True)
    with t3:
        clrs=["#4ecdc4","#45b7d1","#f7dc6f","#e74c3c","#9b59b6","#2ecc71"]
        fig=go.Figure(go.Bar(x=fac_,y=pct_,marker_color=clrs,
            text=[f"{v:.1f}%" for v in pct_],textposition="outside"))
        fig.update_layout(title="Ph\u00e2n r\u00e3 \u0111\u00f3ng g\u00f3p v\u00e0o t\u0103ng tr\u01b0\u1edfng GDP 2020-2025",
            yaxis_title="\u0110\u00f3ng_g\u00f3p (%)",template="plotly_dark",height=420)
        st.plotly_chart(fig,use_container_width=True)
        st.dataframe(pd.DataFrame({"Y\u1ebfu t\u1ed1":fac_,"\u0110\u00f3ng g\u00f3p (%)":np.round(pct_,2)}),use_container_width=True)
    with t4:
        ca,cb,cc,cd=st.columns(4)
        D30=ca.slider("D (KTS/GDP %)",15.0,45.0,30.0,1.0)
        AI30=cb.slider("AI (ng.DN s\u1ed1)",80,150,100,5)
        H30=cc.slider("H (L\u0110 \u0110T %)",28.0,45.0,35.0,1.0)
        Kg=cd.slider("K t\u0103ng %/n\u0103m",3.0,10.0,6.0,0.5)
        K30f=np.array(K_VN)[-1]*(1+Kg/100)**5; L30f=np.array(L_VN)[-1]*(1.01**5)
        Y30f=float(Ab_*(1.012**5)*K30f**0.33*L30f**0.42*D30**0.10*AI30**0.08*H30**0.07)
        st.metric("\U0001f52e GDP D\u1ef1 b\u00e1o 2030",f"{Y30f:,.1f} ng.t\u1ef7 VND",delta=f"+{Y30f-Y_VN[-1]:,.1f} so v\u1edbi 2025")
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=YEARS,y=Y_VN,name="L\u1ecbch s\u1eed",line=dict(color=CLR["r"],width=2),mode="lines+markers"))
        fig.add_trace(go.Scatter(x=[2025,2030],y=[Y_VN[-1],Y30f],name="D\u1ef1 b\u00e1o",line=dict(color=CLR["y"],width=2,dash="dot"),mode="lines+markers"))
        fig.update_layout(title="L\u1ed9 tr\u00ecnh GDP 2020\u21922030",template="plotly_dark",height=380)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b2":
    st.title("\U0001f4b0 B\u00e0i 2 \u2014 LP Ph\u00e2n b\u1ed5 ng\u00e2n s\u00e1ch \u0111\u1ea7u t\u01b0 s\u1ed1")
    z_,al_,bgs_,zv_,shad_=m3_data()
    cats=["H\u1ea1 t\u1ea7ng s\u1ed1","AI & D\u1eef li\u1ec7u","Nh\u00e2n l\u1ef1c s\u1ed1","R&D c\u00f4ng ngh\u1ec7"]
    c1,c2,c3=st.columns(3)
    c1.metric("Z* GDP t\u0103ng th\u00eam",f"{z_:.2f} ng.t\u1ef7")
    c2.metric("Ng\u00e2n s\u00e1ch d\u00f9ng",f"{sum(al_):.0f}/100 ng.t\u1ef7")
    c3.metric("Shadow Price \u2202Z/\u2202B","~1.35 ng.t\u1ef7/ng.t\u1ef7")
    st.divider()
    t1,t2,t3,t4=st.tabs(["\U0001f4ca Ph\u00e2n b\u1ed5 t\u1ed1i \u01b0u","\U0001f4a1 Shadow Price","\U0001f4c9 \u0110\u1ed9 nh\u1ea1y ng\u00e2n s\u00e1ch","\U0001f504 K\u1ecbch b\u1ea3n x3\u226530"])
    with t1:
        fig=go.Figure(go.Bar(x=cats,y=al_,marker_color=[CLR["b"],CLR["r"],CLR["g"],CLR["y"]],
            text=[f"{v:.1f}" for v in al_],textposition="outside"))
        fig.update_layout(title="Ph\u00e2n b\u1ed5 ng\u00e2n s\u00e1ch t\u1ed1i \u01b0u (ng.t\u1ef7 VND)",
            yaxis_title="ng.t\u1ef7 VND",template="plotly_dark",height=420)
        st.plotly_chart(fig,use_container_width=True)
        st.dataframe(pd.DataFrame({"H\u1ea1ng m\u1ee5c":cats,"Ph\u00e2n b\u1ed5":np.round(al_,2),
            "T\u1ef7 l\u1ec7 (%)":np.round(np.array(al_)/sum(al_)*100,1),
            "H\u1ec7 s\u1ed1 GDP":[0.85,1.20,0.95,1.35]}),use_container_width=True)
    with t2:
        sn=["T\u1ed5ng NS","H\u1ea1 t\u1ea7ng min","AI min","Nh\u00e2n l\u1ef1c min","R&D min","CN chi\u1ebfn l\u01b0\u1ee3c"]
        sv=shad_; clrs_s=["#2ecc71" if v>=0 else "#e74c3c" for v in sv]
        fig=go.Figure(go.Bar(x=sn,y=sv,marker_color=clrs_s,
            text=[f"{v:.2f}" for v in sv],textposition="outside"))
        fig.update_layout(title="Gi\u00e1 \u0111\u1ed1i ng\u1eabu (Shadow Price) c\u00e1c r\u00e0ng bu\u1ed9c",
            template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)
        st.info("\U0001f4a1 Shadow Price T\u1ed5ng NS = **1.35**: t\u0103ng 1 ng.t\u1ef7 ng\u00e2n s\u00e1ch \u2192 GDP t\u0103ng th\u00eam 1.35 ng.t\u1ef7.")
    with t3:
        fig=go.Figure(go.Scatter(x=bgs_,y=zv_,mode="lines+markers+text",
            text=[str(v) for v in zv_],textposition="top center",
            line=dict(color=CLR["g"],width=3),marker=dict(size=8)))
        fig.update_layout(title="\u0110\u01b0\u1eddng cong Z*(B) \u2014 \u0110\u1ed9 nh\u1ea1y ng\u00e2n s\u00e1ch",
            xaxis_title="Ng\u00e2n s\u00e1ch (ng.t\u1ef7)",yaxis_title="GDP Gain Z*",
            template="plotly_dark",height=420)
        st.plotly_chart(fig,use_container_width=True)
    with t4:
        z_new=m3b_data(100)
        if z_new:
            st.metric("Z* k\u1ecbch b\u1ea3n x3\u226530",f"{z_new} ng.t\u1ef7",delta=f"{round(z_new-z_,2)}")
            fig=go.Figure(go.Bar(x=["G\u1ed1c (x3\u226520)","K\u1ecbch b\u1ea3n (x3\u226530)"],y=[z_,z_new],
                marker_color=[CLR["b"],CLR["r"]],text=[f"{z_}",f"{z_new}"],textposition="outside"))
            fig.update_layout(title="So s\u00e1nh GDP Gain",template="plotly_dark",height=350)
            st.plotly_chart(fig,use_container_width=True)

elif pg=="b3":
    st.title("\U0001f3ed B\u00e0i 3 \u2014 Ch\u1ec9 s\u1ed1 \u01b0u ti\u00ean Priority 10 ng\u00e0nh")
    pri_,pg_,pi_=priority_data()
    idx=np.argsort(pri_)[::-1]
    c1,c2=st.columns(2)
    c1.metric("Ng\u00e0nh \u01b0u ti\u00ean #1",SECTORS10[idx[0]],f"{pri_[idx[0]]:.3f}")
    c2.metric("Ng\u00e0nh \u01b0u ti\u00ean #2",SECTORS10[idx[1]],f"{pri_[idx[1]]:.3f}")
    st.divider()
    t1,t2,t3=st.tabs(["\U0001f4ca X\u1ebfp h\u1ea1ng Priority","\U0001f525 Sensitivity a6","\U0001f4cb So s\u00e1nh 2 b\u1ed9 tr\u1ecdng s\u1ed1"])
    with t1:
        df3=pd.DataFrame({"Ng\u00e0nh":SECTORS10,"Priority":np.round(pri_,4)}).sort_values("Priority",ascending=False)
        df3["Rank"]=range(1,11)
        fig=go.Figure(go.Bar(x=df3["Priority"].values,y=df3["Ng\u00e0nh"].values,orientation="h",
            marker=dict(color=df3["Priority"].values,colorscale="Plasma"),
            text=df3["Priority"].round(3).astype(str).values,textposition="outside"))
        fig.update_layout(title="Ch\u1ec9 s\u1ed1 Priority 10 ng\u00e0nh Vi\u1ec7t Nam 2024",
            template="plotly_dark",height=460)
        st.plotly_chart(fig,use_container_width=True)
        st.dataframe(df3[["Rank","Ng\u00e0nh","Priority"]],use_container_width=True)
    with t2:
        st.info("Top-3 ng\u00e0nh theo tr\u1ecdng s\u1ed1 a6 (AI Readiness) thay \u0111\u1ed5i t\u1eeb 0.05\u21920.40:")
        gr_=np.array([3.27,9.64,7.45,-1.20,7.10,7.36,9.93,7.85,6.42,6.85])
        pr_=np.array([103.4,241.2,168.8,1290.5,145.3,1072.4,321.4,713.8,205.7,437.1])
        sp_=np.array([0.35,0.78,0.42,0.30,0.55,0.85,0.72,0.92,0.65,0.60])
        ex_=np.array([40.5,290.9,2.5,8.2,5.5,1.2,3.1,178.0,0.0,0.0])
        lb_=np.array([13.20,11.50,4.80,0.30,7.80,0.55,1.95,0.62,2.15,0.75])
        ai_=np.array([15,55,20,30,48,72,42,88,38,45])
        rk_=np.array([18,42,25,55,38,52,35,28,22,18])
        ng_=lambda x:(x-x.min())/(x.max()-x.min()+1e-12)
        nb_=lambda x:(x.max()-x)/(x.max()-x.min()+1e-12)
        Xg_=np.column_stack([ng_(gr_),ng_(pr_),ng_(sp_),ng_(ex_),ng_(lb_),ng_(ai_)])
        Xb_=nb_(rk_)
        rows_h=[]
        for a6v in np.arange(0.05,0.45,0.05):
            w_=np.array([0.15,0.15,0.20,0.15,0.10,a6v]); w_=w_/w_.sum()*(1-0.15)
            p_=Xg_@w_-0.15*Xb_
            t3_=[SECTORS10[i] for i in np.argsort(p_)[::-1][:3]]
            rows_h.append({"a6":round(a6v,2),"#1":t3_[0],"#2":t3_[1],"#3":t3_[2]})
        st.dataframe(pd.DataFrame(rows_h),use_container_width=True)
    with t3:
        fig=go.Figure()
        fig.add_trace(go.Bar(name="T\u0103ng tr\u01b0\u1edfng",x=SECTORS10,y=pg_,marker_color=CLR["b"]))
        fig.add_trace(go.Bar(name="Bao tr\u00f9m",x=SECTORS10,y=pi_,marker_color=CLR["g"]))
        fig.update_layout(barmode="group",title="So s\u00e1nh 2 b\u1ed9 tr\u1ecdng s\u1ed1 ch\u00ednh s\u00e1ch",
            template="plotly_dark",height=420,xaxis_tickangle=-30)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b4":
    st.title("\U0001f5fa\ufe0f B\u00e0i 4 \u2014 LP Ph\u00e2n b\u1ed5 ng\u00e2n s\u00e1ch s\u1ed1 theo ng\u00e0nh-v\u00f9ng")
    X_opt,z4=m4_lp_data()
    items4=["I","D","AI","H"]
    c1,c2=st.columns(2)
    c1.metric("Z* GDP Gain",f"{z4:,.0f} t\u1ef7 VND")
    c2.metric("Ng\u00e2n s\u00e1ch t\u1ed5ng","50,000 t\u1ef7 VND")
    st.divider()
    t1,t2=st.tabs(["\U0001f5fa\ufe0f Heatmap ph\u00e2n b\u1ed5","\U0001f4ca Bi\u1ec3u \u0111\u1ed3 v\u00f9ng"])
    with t1:
        fig=px.imshow(np.round(X_opt,0),x=items4,y=REGIONS,color_continuous_scale="Blues",
            text_auto=True,labels=dict(color="T\u1ef7 VND"))
        fig.update_layout(title="Ph\u00e2n b\u1ed5 t\u1ed1i \u01b0u X_{j,r} \u2014 6 v\u00f9ng \u00d7 4 h\u1ea1ng m\u1ee5c",
            template="plotly_dark",height=420)
        st.plotly_chart(fig,use_container_width=True)
    with t2:
        totals=X_opt.sum(axis=1)
        fig=go.Figure(go.Bar(x=REGIONS,y=np.round(totals,0),
            marker_color=px.colors.qualitative.Plotly[:6],
            text=np.round(totals,0),textposition="outside"))
        fig.update_layout(title="T\u1ed5ng ng\u00e2n s\u00e1ch m\u1ed7i v\u00f9ng",yaxis_title="T\u1ef7 VND",
            template="plotly_dark",height=380)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b5":
    st.title("\U0001f522 B\u00e0i 5 \u2014 MIP L\u1ef1a ch\u1ecdn 15 d\u1ef1 \u00e1n")
    sel_,zval_,nm_,C_,B_=m5_mip_data()
    c1,c2,c3=st.columns(3)
    c1.metric("T\u1ed5ng NPV",f"{zval_:,.0f} t\u1ef7")
    c2.metric("D\u1ef1 \u00e1n ch\u1ecdn",f"{len(sel_)}/15")
    c3.metric("T\u1ed5ng chi ph\u00ed",f"{sum(C_.get(i,0) for i in sel_):,} t\u1ef7")
    st.divider()
    t1,t2=st.tabs(["\u2705 D\u1ef1 \u00e1n ch\u1ecdn","\U0001f4ca Cost-Benefit"])
    with t1:
        rows5=[{"M\u00e3":f"P{i}","T\u00ean":nm_.get(i,f"P{i}"),
                "Chi ph\u00ed (t\u1ef7)":C_.get(i,0),"NPV (t\u1ef7)":B_.get(i,0),
                "ROI":round(B_.get(i,0)/max(C_.get(i,1),1),2)} for i in sel_]
        st.dataframe(pd.DataFrame(rows5),use_container_width=True)
        st.success(f"\u2705 {len(sel_)} d\u1ef1 \u00e1n, T\u1ed5ng NPV={zval_:,.0f} t\u1ef7 VND")
    with t2:
        all_p=list(nm_.keys())
        fig=go.Figure()
        fig.add_trace(go.Bar(name="Chi ph\u00ed",x=[nm_[i] for i in all_p],y=[C_.get(i,0) for i in all_p],marker_color=CLR["b"]))
        fig.add_trace(go.Bar(name="NPV",x=[nm_[i] for i in all_p],y=[B_.get(i,0) for i in all_p],marker_color=CLR["g"]))
        fig.update_layout(barmode="group",title="Chi ph\u00ed vs NPV c\u00e1c d\u1ef1 \u00e1n",
            template="plotly_dark",height=420,xaxis_tickangle=-30)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b6":
    st.title("\U0001f3c6 B\u00e0i 6 \u2014 TOPSIS X\u1ebfp h\u1ea1ng 6 v\u00f9ng")
    se_,sn_,went_=m2_data()
    crit_n=["GRDP/ng\u01b0\u1eddi","FDI","Digital Index","AI Readiness","L\u0110 \u0110T","R&D","Internet","Gini"]
    t1,t2,t3=st.tabs(["\U0001f3c6 Expert","\U0001f52c Entropy","\U0001f4ca So s\u00e1nh"])
    with t1:
        df6e=pd.DataFrame({"V\u00f9ng":REGIONS,"Score":np.round(se_,4)}).sort_values("Score",ascending=False)
        df6e["Rank"]=range(1,7)
        fig=go.Figure(go.Bar(x=df6e["Score"],y=df6e["V\u00f9ng"],orientation="h",
            marker=dict(color=df6e["Score"].values,colorscale="Viridis"),
            text=df6e["Score"].round(3).astype(str),textposition="outside"))
        fig.update_layout(title="TOPSIS Score \u2014 Tr\u1ecdng s\u1ed1 chuy\u00ean gia",template="plotly_dark",height=380)
        st.plotly_chart(fig,use_container_width=True)
        st.dataframe(df6e,use_container_width=True)
    with t2:
        df6n=pd.DataFrame({"V\u00f9ng":REGIONS,"Score":np.round(sn_,4)}).sort_values("Score",ascending=False)
        df6n["Rank"]=range(1,7)
        fig=go.Figure(go.Bar(x=df6n["Score"],y=df6n["V\u00f9ng"],orientation="h",
            marker=dict(color=df6n["Score"].values,colorscale="Plasma"),
            text=df6n["Score"].round(3).astype(str),textposition="outside"))
        fig.update_layout(title="TOPSIS Score \u2014 Entropy Weights",template="plotly_dark",height=380)
        st.plotly_chart(fig,use_container_width=True)
        st.dataframe(pd.DataFrame({"Ti\u00eau ch\u00ed":crit_n,"Entropy W":np.round(went_,4)}),use_container_width=True)
    with t3:
        fig=go.Figure()
        fig.add_trace(go.Bar(name="Expert",x=REGIONS,y=np.round(se_,4),marker_color=CLR["b"]))
        fig.add_trace(go.Bar(name="Entropy",x=REGIONS,y=np.round(sn_,4),marker_color=CLR["r"]))
        fig.update_layout(barmode="group",title="TOPSIS: Expert vs Entropy",template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b7":
    st.title("\U0001f310 B\u00e0i 7 \u2014 T\u1ed1i \u01b0u \u0111a m\u1ee5c ti\u00eau Pareto (NSGA-II)")
    dfp_,bidx_=pareto_data()
    c1,c2,c3,c4=st.columns(4)
    c1.metric("GDP max",f"{dfp_['GDP'].max():.1f}")
    c2.metric("Gini (GDP max)",f"{dfp_.loc[dfp_['GDP'].idxmax(),'Gini']:.0f}")
    c3.metric("CO2 (th\u1ecfa hi\u1ec7p)",f"{dfp_.loc[bidx_,'CO2']:.1f}")
    c4.metric("Cyber (th\u1ecfa hi\u1ec7p)",f"{dfp_.loc[bidx_,'Cyber']:.1f}")
    st.divider()
    t1,t2,t3=st.tabs(["\U0001f4d0 Pareto 2D","\U0001f30d Scatter 3D","\U0001f4ca Parallel Coords"])
    with t1:
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=dfp_["GDP"],y=dfp_["Gini"],mode="markers",
            marker=dict(color=dfp_["CO2"],colorscale="Viridis",size=7,
            colorbar=dict(title="CO2")),name="Pareto"))
        fig.add_trace(go.Scatter(x=[dfp_.loc[bidx_,"GDP"]],y=[dfp_.loc[bidx_,"Gini"]],
            mode="markers",marker=dict(color="red",size=14,symbol="star"),name="Th\u1ecfa hi\u1ec7p"))
        fig.update_layout(title="\u0110\u01b0\u1eddng bi\u00ean Pareto: GDP Gain vs Gini",
            xaxis_title="f1 (GDP)",yaxis_title="f2 (Gini)",template="plotly_dark",height=430)
        st.plotly_chart(fig,use_container_width=True)
    with t2:
        fig=go.Figure(data=go.Scatter3d(x=dfp_["GDP"],y=dfp_["Gini"],z=dfp_["CO2"],
            mode="markers",marker=dict(size=4,color=dfp_["Cyber"],colorscale="Rainbow",
            colorbar=dict(title="Cyber"))))
        fig.update_layout(title="Scatter 3D: f1 vs f2 vs f3",
            scene=dict(xaxis_title="GDP",yaxis_title="Gini",zaxis_title="CO2"),
            template="plotly_dark",height=480)
        st.plotly_chart(fig,use_container_width=True)
    with t3:
        fig=px.parallel_coordinates(dfp_,dimensions=["GDP","Gini","CO2","Cyber"],
            color="GDP",color_continuous_scale="Plasma")
        fig.update_layout(title="Parallel Coordinates \u2014 4 m\u1ee5c ti\u00eau",template="plotly_dark",height=430)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b8":
    st.title("\U0001f4c8 B\u00e0i 8 \u2014 T\u1ed1i \u01b0u \u0111\u1ed9ng 2026-2035")
    yrs_,Ks_,Ds_,AIs_,Hs_,Ys_,Cs_=dynamic_data()
    c1,c2,c3=st.columns(3)
    c1.metric("GDP 2026",f"{Ys_[0]:,.0f} ng.t\u1ef7")
    c2.metric("GDP 2035",f"{Ys_[-1]:,.0f} ng.t\u1ef7")
    c3.metric("T\u0103ng tr\u01b0\u1edfng TB",f"{((Ys_[-1]/Ys_[0])**(1/9)-1)*100:.1f}%/n\u0103m")
    st.divider()
    t1,t2,t3=st.tabs(["\U0001f4c8 Qu\u1ef9 \u0111\u1ea1o Y&C","\U0001f4ca V\u1ed1n c\u00e1c lo\u1ea1i","\u26a1 C\u00fa s\u1ed1c 2028"])
    with t1:
        fig=make_subplots(rows=1,cols=2,subplot_titles=("S\u1ea3n l\u01b0\u1ee3ng Y_t","Ti\u00eau d\u00f9ng C_t"))
        fig.add_trace(go.Scatter(x=yrs_,y=Ys_,name="Y",line=dict(color=CLR["r"],width=2)),row=1,col=1)
        fig.add_trace(go.Scatter(x=yrs_,y=Cs_,name="C",line=dict(color=CLR["g"],width=2)),row=1,col=2)
        fig.update_layout(template="plotly_dark",height=380,title="Qu\u1ef9 \u0111\u1ea1o t\u1ed1i \u01b0u Y v\u00e0 C")
        st.plotly_chart(fig,use_container_width=True)
    with t2:
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=yrs_,y=Ks_,name="K (V\u1ed1n)",mode="lines+markers"))
        fig.add_trace(go.Scatter(x=yrs_,y=[d*500 for d in Ds_],name="D\u00d7500",mode="lines+markers"))
        fig.add_trace(go.Scatter(x=yrs_,y=AIs_,name="AI (ng.DN)",mode="lines+markers"))
        fig.add_trace(go.Scatter(x=yrs_,y=[h*100 for h in Hs_],name="H\u00d7100",mode="lines+markers"))
        fig.update_layout(title="Qu\u1ef9 \u0111\u1ea1o c\u00e1c lo\u1ea1i v\u1ed1n 2026-2035",template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)
    with t3:
        shock_Y=[y if i!=2 else y*0.92 for i,y in enumerate(Ys_)]
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=yrs_,y=Ys_,name="G\u1ed1c",line=dict(color=CLR["b"],width=2)))
        fig.add_trace(go.Scatter(x=yrs_,y=shock_Y,name="C\u00fa s\u1ed1c 2028 (-8%)",
            line=dict(color=CLR["r"],width=2,dash="dot")))
        fig.update_layout(title="Ph\u00e2n t\u00edch c\u00fa s\u1ed1c Y_2028 gi\u1ea3m 8%",template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b9":
    st.title("\U0001f465 B\u00e0i 9 \u2014 T\u00e1c \u0111\u1ed9ng AI t\u1edbi lao \u0111\u1ed9ng Vi\u1ec7t Nam")
    df9_=labor_data()
    c1,c2,c3=st.columns(3)
    c1.metric("T\u1ed5ng NetJob",f"{df9_['NetJob (k)'].sum():.1f}k")
    c2.metric("Ng\u00e0nh t\u1ea1o nhi\u1ec1u nh\u1ea5t",df9_.loc[df9_['NetJob (k)'].idxmax(),'Ng\u00e0nh'])
    c3.metric("Ng\u00e0nh r\u1ee7i ro cao",df9_.loc[df9_['Risk (%)'].idxmax(),'Ng\u00e0nh'])
    st.divider()
    t1,t2,t3=st.tabs(["\U0001f4ca NetJob","\u26a0\ufe0f Luồng việc làm","\U0001f5c3\ufe0f B\u1ea3ng chi ti\u1ebft"])
    with t1:
        clrs9=["#2ecc71" if v>0 else "#e74c3c" for v in df9_["NetJob (k)"]]
        fig=go.Figure(go.Bar(x=df9_["Ng\u00e0nh"],y=df9_["NetJob (k)"],
            marker_color=clrs9,text=df9_["NetJob (k)"],textposition="outside"))
        fig.add_hline(y=0,line_color="white",line_dash="dash")
        fig.update_layout(title="NetJob r\u00f2ng theo ng\u00e0nh (ngh\u00ecn vi\u1ec7c l\u00e0m)",
            template="plotly_dark",height=430)
        st.plotly_chart(fig,use_container_width=True)
    with t2:
        fig=go.Figure()
        fig.add_trace(go.Bar(name="Vi\u1ec7c l\u00e0m m\u1edbi",x=df9_["Ng\u00e0nh"],y=df9_["New (k)"],marker_color=CLR["g"]))
        fig.add_trace(go.Bar(name="N\u00e2ng c\u1ea5p",x=df9_["Ng\u00e0nh"],y=df9_["Upgrade (k)"],marker_color=CLR["b"]))
        fig.add_trace(go.Bar(name="D\u1ecbch chuy\u1ec3n",x=df9_["Ng\u00e0nh"],y=-df9_["Displaced (k)"],marker_color=CLR["r"]))
        fig.update_layout(barmode="relative",title="Lu\u1ed3ng vi\u1ec7c l\u00e0m theo ng\u00e0nh",template="plotly_dark",height=430)
        st.plotly_chart(fig,use_container_width=True)
    with t3:
        st.dataframe(df9_,use_container_width=True)

elif pg=="b10":
    st.title("\U0001f3b2 B\u00e0i 10 \u2014 Quy ho\u1ea1ch ng\u1eabu nhi\u00ean hai giai \u0111o\u1ea1n")
    xopt_,L10_,zsp_,zev_,vss_,evpi_,gdpsc_=stochastic_data()
    c1,c2,c3,c4=st.columns(4)
    c1.metric("Z* Stochastic",f"{zsp_:,.0f}")
    c2.metric("Z* EV (x\u00e1c \u0111\u1ecbnh)",f"{zev_:,.0f}")
    c3.metric("VSS",f"{vss_:,.0f}")
    c4.metric("EVPI",f"{evpi_:,.0f}")
    st.divider()
    t1,t2,t3=st.tabs(["\U0001f4ca First-stage","\U0001f30d GDP/K\u1ecbch b\u1ea3n","\U0001f4cb B\u1ea3ng \u03b2"])
    with t1:
        fig=go.Figure(go.Bar(x=L10_,y=xopt_,
            marker_color=[CLR["b"],CLR["g"],CLR["r"],CLR["y"]],
            text=[f"{v:,.0f}" for v in xopt_],textposition="outside"))
        fig.update_layout(title="Ph\u00e2n b\u1ed5 First-stage t\u1ed1i \u01b0u (t\u1ef7 VND)",
            template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)
        st.info(f"VSS={vss_:,.0f} > 0 \u2192 C\u00e2n nh\u1eafc b\u1ea5t \u0111\u1ecbnh c\u00f3 gi\u00e1 tr\u1ecb!")
    with t2:
        scens10=["s1 L\u1ea1c quan (30%)","s2 C\u01a1 s\u1edf (45%)","s3 Bi quan (20%)","s4 KH (5%)"]
        fig=go.Figure(go.Bar(x=scens10,y=gdpsc_,
            marker_color=[CLR["g"],CLR["b"],CLR["y"],CLR["r"]],
            text=gdpsc_,textposition="outside"))
        fig.update_layout(title="GDP Gain theo k\u1ecbch b\u1ea3n",template="plotly_dark",height=400)
        st.plotly_chart(fig,use_container_width=True)
    with t3:
        bd=pd.DataFrame({"H\u1ea1ng m\u1ee5c":L10_,
            "\u03b2 c\u01a1 b\u1ea3n":[1.00,1.10,1.25,0.95],
            "\u03b2 s1 L\u1ea1c quan":[1.25,1.35,1.55,1.05],
            "\u03b2 s2 C\u01a1 s\u1edf":[1.00,1.10,1.25,0.95],
            "\u03b2 s3 Bi quan":[0.75,0.85,0.90,1.00],
            "\u03b2 s4 KH":[0.40,0.50,0.55,1.10]})
        st.dataframe(bd,use_container_width=True)

elif pg=="b11":
    st.title("\U0001f916 B\u00e0i 11 \u2014 Q-Learning RL Ch\u00ednh s\u00e1ch kinh t\u1ebf")
    eps_,rw_,ba_,pp_=rl_data()
    c1,c2=st.columns(2)
    c1.metric("\u03c0* Reward TB",f"{pp_[list(pp_.keys())[0]]:.2f}")
    c2.metric("C\u1ea3i thi\u1ec7n vs Random",f"+{pp_[list(pp_.keys())[0]]-pp_['Random']:.2f}")
    st.divider()
    t1,t2,t3=st.tabs(["\U0001f4c8 Learning Curve","\U0001f3af Ch\u00ednh s\u00e1ch \u03c0*(s)","\U0001f4ca So s\u00e1nh"])
    with t1:
        fig=go.Figure(go.Scatter(x=eps_,y=rw_,name="Q-Learning",
            line=dict(color=CLR["y"],width=2)))
        for pol,val in pp_.items():
            if "RL" not in pol and "\u03c0" not in pol:
                fig.add_hline(y=val,line_dash="dot",annotation_text=pol,line_color="gray")
        fig.update_layout(title="\u0110\u01b0\u1eddng cong h\u1ecdc t\u1eadp Q-Learning",
            xaxis_title="Episode",yaxis_title="Ph\u1ea7n th\u01b0\u1edfng",
            template="plotly_dark",height=420)
        st.plotly_chart(fig,use_container_width=True)
    with t2:
        df11=pd.DataFrame([{"Tr\u1ea1ng th\u00e1i":k,"H\u00e0nh \u0111\u1ed9ng t\u1ed1i \u01b0u \u03c0*(s)":v} for k,v in ba_.items()])
        st.dataframe(df11,use_container_width=True)
        st.info("\U0001f4a1 VN 2026: \u03c0* ch\u1ecdn **a1: C\u00e2n b\u1eb1ng** \u2014 ph\u00f9 h\u1ee3p giai \u0111o\u1ea1n chuy\u1ec3n giao.")
    with t3:
        pol_=list(pp_.keys()); vals_=list(pp_.values())
        clrs11=["#2ecc71" if "\u03c0" in p or "RL" in p else "#60a5fa" for p in pol_]
        fig=go.Figure(go.Bar(x=pol_,y=vals_,marker_color=clrs11,
            text=[f"{v:.2f}" for v in vals_],textposition="outside"))
        fig.update_layout(title="So s\u00e1nh hi\u1ec7u qu\u1ea3 c\u00e1c ch\u00ednh s\u00e1ch",
            template="plotly_dark",height=380)
        st.plotly_chart(fig,use_container_width=True)

elif pg=="b12":
    st.title("\U0001f1fb\U0001f1f3 B\u00e0i 12 \u2014 AIDEOM-VN: H\u1ec7 th\u1ed1ng t\u00edch h\u1ee3p")
    st.markdown("*T\u1ed5ng h\u1ee3p 6 module M1-M6 th\u00e0nh dashboard h\u1ed7 tr\u1ee3 ra quy\u1ebft \u0111\u1ecbnh*")
    A__,Ab__,Yh__,mape__,fac__,pct__,gY__,Y30__=m1_data()
    se__,sn__,_=m2_data()
    z3__,al3__,bgs3__,zv3__,_=m3_data()
    df9__=labor_data()
    M6__=scenarios_m12()
    dfp__,bidx__=pareto_data()

    tov,tph,tsc,tri=st.tabs([
        "\U0001f5fa\ufe0f T\u1ed5ng quan (M1-M2)","\U0001f4b0 Ph\u00e2n b\u1ed5 (M3)",
        "\U0001f4ca 5 K\u1ecbch b\u1ea3n (M6)","\u26a0\ufe0f C\u1ea3nh b\u00e1o (M4-M5)"
    ])

    with tov:
        st.subheader("M1 \u2014 D\u1ef1 b\u00e1o kinh t\u1ebf (Cobb-Douglas)")
        c1,c2,c3=st.columns(3)
        c1.metric("MAPE",f"{mape__:.2f}%")
        c2.metric("\u0100",f"{Ab__:.4f}")
        c3.metric("Y 2030 d\u1ef1 b\u00e1o",f"{Y30__:,.0f} ng.t\u1ef7")
        clrs12=["#4ecdc4","#45b7d1","#f7dc6f","#e74c3c","#9b59b6","#2ecc71"]
        fig=go.Figure(go.Bar(x=fac__,y=pct__,marker_color=clrs12,
            text=[f"{v:.1f}%" for v in pct__],textposition="outside"))
        fig.update_layout(title="Ph\u00e2n r\u00e3 \u0111\u00f3ng g\u00f3p t\u0103ng tr\u01b0\u1edfng 2020-2025",
            yaxis_title="\u0110\u00f3ng_g\u00f3p_pct",template="plotly_dark",height=360)
        st.plotly_chart(fig,use_container_width=True)
        st.subheader("M2 \u2014 \u0110\u00e1nh gi\u00e1 s\u1eb5n s\u00e0ng s\u1ed1 (TOPSIS)")
        c1,c2=st.columns(2)
        df_m2_=pd.DataFrame({"V\u00f9ng":REGIONS,"TOPSIS Expert":np.round(se__,4),
            "TOPSIS Entropy":np.round(sn__,4)}).sort_values("TOPSIS Expert",ascending=False)
        with c1:
            fig=go.Figure(go.Bar(x=df_m2_["TOPSIS Expert"],y=df_m2_["V\u00f9ng"],orientation="h",
                marker=dict(color=df_m2_["TOPSIS Expert"].values,colorscale="Viridis"),
                text=df_m2_["TOPSIS Expert"].round(3).astype(str),textposition="outside"))
            fig.update_layout(title="Expert",template="plotly_dark",height=300)
            st.plotly_chart(fig,use_container_width=True)
        with c2:
            fig=go.Figure(go.Bar(x=df_m2_["TOPSIS Entropy"],y=df_m2_["V\u00f9ng"],orientation="h",
                marker=dict(color=df_m2_["TOPSIS Entropy"].values,colorscale="Plasma"),
                text=df_m2_["TOPSIS Entropy"].round(3).astype(str),textposition="outside"))
            fig.update_layout(title="Entropy",template="plotly_dark",height=300)
            st.plotly_chart(fig,use_container_width=True)

    with tph:
        st.subheader("M3 \u2014 T\u1ed1i \u01b0u ph\u00e2n b\u1ed5 ng\u00e2n s\u00e1ch (LP)")
        cats3_=["H\u1ea1 t\u1ea7ng s\u1ed1","AI & D\u1eef li\u1ec7u","Nh\u00e2n l\u1ef1c s\u1ed1","R&D"]
        c1,c2=st.columns(2)
        with c1:
            fig=go.Figure(go.Pie(labels=cats3_,values=al3__,hole=0.45,
                marker_colors=[CLR["b"],CLR["r"],CLR["g"],CLR["y"]]))
            fig.update_layout(title=f"Ph\u00e2n b\u1ed5 t\u1ed1i \u01b0u (Z*={z3__})",
                template="plotly_dark",height=340)
            st.plotly_chart(fig,use_container_width=True)
        with c2:
            fig=go.Figure(go.Scatter(x=bgs3__,y=zv3__,mode="lines+markers",
                line=dict(color=CLR["g"],width=2),text=[str(v) for v in zv3__],textposition="top center"))
            fig.update_layout(title="\u0110\u1ed9 nh\u1ea1y ng\u00e2n s\u00e1ch Z*(B)",
                template="plotly_dark",height=340)
            st.plotly_chart(fig,use_container_width=True)

    with tsc:
        st.subheader("M6 \u2014 So s\u00e1nh 5 k\u1ecbch b\u1ea3n ch\u00ednh s\u00e1ch")
        bgdp=M6__.loc[M6__["GDP Gain (t\u1ef7)"].idxmax(),"K\u1ecbch b\u1ea3n"]
        bjob=M6__.loc[M6__["NetJob (k)"].idxmax(),"K\u1ecbch b\u1ea3n"]
        benv=M6__.loc[M6__["CO2 Risk"].idxmin(),"K\u1ecbch b\u1ea3n"]
        beq=M6__.loc[M6__["Equity"].idxmax(),"K\u1ecbch b\u1ea3n"]
        c1,c2,c3,c4=st.columns(4)
        c1.metric("Best GDP",bgdp); c2.metric("Best Jobs",bjob)
        c3.metric("Best CO2",benv); c4.metric("Best Equity",beq)
        fig=go.Figure()
        kpis=["GDP Gain (t\u1ef7)","NetJob (k)","CO2 Risk","Equity"]
        for kpi in kpis:
            vn=M6__[kpi]; mn,mx=vn.min(),vn.max()
            norm=(vn-mn)/(mx-mn+1e-12)
            fig.add_trace(go.Bar(name=kpi,x=M6__["K\u1ecbch b\u1ea3n"],y=norm,
                text=vn.round(1).astype(str),textposition="outside"))
        fig.update_layout(barmode="group",title="So s\u00e1nh KPI 5 k\u1ecbch b\u1ea3n (chu\u1ea9n h\u00f3a)",
            template="plotly_dark",height=420)
        st.plotly_chart(fig,use_container_width=True)
        fig_r=go.Figure()
        for _,row in M6__.iterrows():
            fig_r.add_trace(go.Scatterpolar(
                r=[row["GDP Gain (t\u1ef7)"]/1e5,row["NetJob (k)"]/10,(5-row["CO2 Risk"])/5*100,row["Equity"]],
                theta=["GDP Gain","NetJob","M\u00f4i tr\u01b0\u1eddng","C\u00f4ng b\u1eb1ng"],
                fill="toself",name=row["K\u1ecbch b\u1ea3n"]))
        fig_r.update_layout(polar=dict(radialaxis=dict(visible=True)),
            title="Radar Chart \u2014 5 k\u1ecbch b\u1ea3n",template="plotly_dark",height=460)
        st.plotly_chart(fig_r,use_container_width=True)
        st.dataframe(M6__,use_container_width=True)

    with tri:
        st.subheader("M4 \u2014 M\u00f4 ph\u1ecfng lao \u0111\u1ed9ng")
        clrs9_=["#2ecc71" if v>0 else "#e74c3c" for v in df9__["NetJob (k)"]]
        fig=go.Figure(go.Bar(x=df9__["Ng\u00e0nh"],y=df9__["NetJob (k)"],
            marker_color=clrs9_,text=df9__["NetJob (k)"],textposition="outside"))
        fig.add_hline(y=0,line_color="white",line_dash="dash")
        fig.update_layout(title="NetJob r\u00f2ng M4",template="plotly_dark",height=360)
        st.plotly_chart(fig,use_container_width=True)
        st.subheader("M5 \u2014 \u0110\u00e1nh gi\u00e1 r\u1ee7i ro Pareto")
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=dfp__["GDP"],y=dfp__["CO2"],mode="markers",
            marker=dict(color=dfp__["Cyber"],colorscale="RdYlGn",size=6,
            colorbar=dict(title="Cyber")),name="Pareto"))
        fig.add_trace(go.Scatter(x=[dfp__.loc[bidx__,"GDP"]],y=[dfp__.loc[bidx__,"CO2"]],
            mode="markers",marker=dict(color="red",size=14,symbol="star"),name="Th\u1ecfa hi\u1ec7p"))
        fig.update_layout(title="Pareto M5: GDP vs CO2",template="plotly_dark",height=380)
        st.plotly_chart(fig,use_container_width=True)
        df_rk=df9__[["Ng\u00e0nh","Risk (%)","NetJob (k)"]].copy()
        df_rk["C\u1ea3nh b\u00e1o"]=df_rk["Risk (%)"].apply(
            lambda r:"\U0001f534 Cao" if r>45 else "\U0001f7e1 TB" if r>30 else "\U0001f7e2 Th\u1ea5p")
        st.dataframe(df_rk,use_container_width=True)

# === FOOTER ===
st.divider()
st.markdown(
    "<div style=\'text-align:center;color:#8b949e;font-size:12px;\'>"
    "\U0001f1fb\U0001f1f3 VN AIDEOM-VN \u2014 Mai Huy\u1ec1n Trang \u00b7 FDE3010 \u00b7 \u0110H Kinh t\u1ebf \u2013 \u0110HQGHN \u00b7 2025"
    "</div>",unsafe_allow_html=True)
'''

with open('/content/app.py', 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

import os
sz = os.path.getsize('/content/app.py')
print(f'✅ Đã ghi /content/app.py ({sz:,} bytes)')
print('⏭️  Chạy Cell 4 để khởi động!')

In [ ]:
# ============================================================
# CELL 4 — KHỞI ĐỘNG WEB APP QUA NGROK
# ============================================================
import subprocess, time, threading, sys, os

# 1. Import pyngrok
try:
    from pyngrok import ngrok
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
    from pyngrok import ngrok

# 2. (Tùy chọn) Paste authtoken ngrok của bạn vào đây
#    Lấy miễn phí tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # ← paste token vào đây nếu có
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✅ ngrok token OK')

# 3. Dừng Streamlit cũ
os.system('pkill -f streamlit 2>/dev/null'); time.sleep(2)

# 4. Khởi động Streamlit background
PORT = 8501
log = '/content/st.log'

def run_st():
    with open(log, 'w') as lf:
        subprocess.Popen(
            [sys.executable, '-m', 'streamlit', 'run', '/content/app.py',
             '--server.port', str(PORT),
             '--server.headless', 'true',
             '--server.enableCORS', 'false',
             '--server.enableXsrfProtection', 'false',
             '--server.fileWatcherType', 'none',
             '--logger.level', 'error'],
            stdout=lf, stderr=lf)

threading.Thread(target=run_st, daemon=True).start()
print('⏳ Chờ Streamlit khởi động...')
time.sleep(10)

# 5. Đóng tunnel cũ rồi tạo mới
for t in ngrok.get_tunnels():
    try: ngrok.disconnect(t.public_url)
    except: pass
time.sleep(1)

tunnel = ngrok.connect(PORT, 'http')
url    = tunnel.public_url

# 6. Kiểm tra Streamlit
time.sleep(3)
try:
    import urllib.request
    urllib.request.urlopen(f'http://localhost:{PORT}', timeout=8)
    st_ok = '✅ Streamlit đang chạy'
except:
    st_ok = '⚠️  Streamlit chưa sẵn sàng — thử reload URL sau 10 giây'

print('\n' + '='*60)
print('🚀  AIDEOM-VN WEB APP ĐÃ SẴN SÀNG!')
print('='*60)
print(f'🌐  URL Public : {url}')
print(f'🖥️  Local      : http://localhost:{PORT}')
print(f'{st_ok}')
print('='*60)
print('📌  Lưu ý:')
print('    • Click URL → mở web app')
print('    • Nếu trắng màn hình → đợi 10s rồi reload')
print('    • URL hoạt động ~2 giờ (ngrok free)')
print('    • Thêm NGROK_TOKEN để dùng lâu hơn')
print('='*60)
print(f'\n🔗  {url}')